# Week 22 — Constrained MDPs, PPO-Lagrangian, CPO/FOCOPS/P3O

This is a reproducible **reference workbook**. Many supplied scaffold or check
cells directly call `project.solutions`; executing them checks the reference
demonstration, not your implementation. They do not automatically switch modes
when run interactively or with papermill.

For your own attempt, save a copy, replace the relevant reference wrapper with
your code, and make sure the checks call that code. Predict a boundary case
before running it. For independently checked TODO exercises with no fallback,
start with [the progressive coding projects](../docs/INTERACTIVE_LEARNING.md).

**Teaching companion:** [Worked examples, learning bridges and self-checks](teaching_guide.md).


**Worked examples and derivations:** [Teaching guide](teaching_guide.md) — prerequisite links, analogy boundaries, calculated examples, code connections, and retrieval questions.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import numpy as np, matplotlib.pyplot as plt
from project import solutions as ref

## Exercise 1 ★ — Lagrangian dual step

Derive on paper the projected-gradient-descent update on $\lambda$ for $L(\pi,\lambda) = J_r - \lambda(J_c - d)$.

> **Hint** (try yourself first): descent on this Lagrangian gives $\lambda \leftarrow [\lambda + \eta(J_c - d)]_+$. Sign matters.

In [ ]:
def exercise_1(lam, eta, J_c, d):
    # YOUR CODE HERE — replace with your derivation
    return ref.exercise_1(lam, eta, J_c, d)

In [ ]:
assert exercise_1(0.5, 0.1, 2.0, 1.0) > 0.5
assert exercise_1(0.05, 1.0, 0.0, 1.0) == 0.0
print('ok')

## Exercise 2 ★ — P3O hinge subgradient

Return a subgradient of $\kappa\cdot[J_c - d]_+$ w.r.t.\ $J_c$.

> **Hint**: it is $\kappa$ above $d$, $0$ below $d$, anything in $[0,\kappa]$ at the kink.

In [ ]:
def exercise_2(J_c, d, kappa):
    # YOUR CODE HERE
    return ref.exercise_2(J_c, d, kappa)

In [ ]:
assert exercise_2(2.0, 1.0, 10.0) == 10.0
assert exercise_2(0.5, 1.0, 10.0) == 0.0
print('ok')

## Exercise 3 ★★ — GAE on the cost stream

Implement Generalised Advantage Estimation (Schulman et al. 2016). PPO-Lagrangian calls it twice: once on reward, once on cost.

> **Hint**: iterate from the end; $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ and $A_t = \delta_t + \gamma\lambda A_{t+1}$ when nonterminal.

In [ ]:
def exercise_3(rews, vals, dones, gamma, lam, last_val=0.0):
    # YOUR CODE HERE
    return ref.exercise_3(rews, vals, dones, gamma, lam, last_val)

In [ ]:
rews=[0.1]*10; vals=[0.0]*10; dones=[False]*9+[True]
adv, ret = exercise_3(rews, vals, dones, 0.9, 0.95)
assert adv.shape == (10,)
print('ok')

## Exercise 4 ★★ — FOCOPS closed-form non-parametric update

Derive and implement $\pi^\star(a|s)\propto \pi_{\rm old}(a|s)\exp((A_r - \nu A_c)/\lambda)$.

> **Hint**: log-space + softmax for numerical stability.

In [ ]:
def exercise_4(pi_old, A_r, A_c, lam, nu):
    # YOUR CODE HERE
    return ref.exercise_4(pi_old, A_r, A_c, lam, nu)

In [ ]:
pi = np.full((4,3), 1/3.0)
A_r = np.random.RandomState(0).randn(4,3); A_c = np.random.RandomState(1).randn(4,3)
new = exercise_4(pi, A_r, A_c, 1.0, 0.5)
assert np.allclose(new.sum(1), 1.0); assert (new >= 0).all()
print('ok')

## Exercise 5 ★★ — Empirical CVaR

Approximate lower-tail $\mathrm{CVaR}_\alpha(G)$ from return samples using the supplied top-k convention; distinguish it from upper-tail cost risk.

> **Hint**: sort, take the worst $\lceil \alpha N \rceil$, average.

In [ ]:
def exercise_5(samples, alpha):
    # YOUR CODE HERE
    return ref.exercise_5(samples, alpha)

In [ ]:
assert abs(exercise_5(np.array([0,1,2,3,4.0]), 0.4) - 0.5) < 1e-9
print('ok')

## Exercise 6 ★★ — Meaningful objective: train PPO-Lagrangian and audit

Train the agent on `LavaGridWorld` and verify discounted cost stays near budget. This is the *matching / safety meaningful objective* — in a real synthetic insurance or synthetic ATS deployment, the cost signal would be a fairness or capacity-saturation metric.

> **Hint**: call `ref.exercise_6(n_updates=20, cost_limit=1.5)`.

In [ ]:
def exercise_6(seed=0, n_updates=20, cost_limit=1.5):
    # YOUR CODE HERE: train and return a dict {history, final_cost, final_return}.
    return ref.exercise_6(seed=seed, n_updates=n_updates, cost_limit=cost_limit)

In [ ]:
out = exercise_6(seed=0, n_updates=20, cost_limit=1.5)
h = out['history']
fig, axes = plt.subplots(1,2, figsize=(10,3.5))
axes[0].plot(h['return'], color=RL_PALETTE['blue']); axes[0].set_title('Return')
axes[1].plot(h['cost'], color=RL_PALETTE['red']); axes[1].axhline(1.5, color=RL_PALETTE['gray'], linestyle=':')
axes[1].set_title('Cost vs budget'); plt.tight_layout(); plt.show()
print('final cost', out['final_cost'], 'final return', out['final_return'])

## Exercise 7 ★★★ — Open-ended: safe vs unsafe

Train with $d = 0.5$ vs $d = 10^6$. Plot return and cost for both. Explain (markdown) why the safe policy may have lower return — and why this is *acceptable*.

> **Hint**: `ref.exercise_7` returns the two histories.

In [ ]:
def exercise_7(seed=0, n_updates=20):
    # YOUR CODE HERE
    return ref.exercise_7(seed=seed, n_updates=n_updates)

In [ ]:
res = exercise_7(seed=0, n_updates=20)
fig, axes = plt.subplots(1,2, figsize=(10,3.5))
axes[0].plot(res['unsafe']['return'], label='unconstrained', color=RL_PALETTE['gray'])
axes[0].plot(res['safe']['return'], label='d=0.5', color=RL_PALETTE['blue']); axes[0].legend(); axes[0].set_title('Return')
axes[1].plot(res['unsafe']['cost'], label='unconstrained', color=RL_PALETTE['gray'])
axes[1].plot(res['safe']['cost'], label='d=0.5', color=RL_PALETTE['red']); axes[1].legend(); axes[1].set_title('Cost')
plt.tight_layout(); plt.show()

## Exercise 8 ★★★ — PID-Lagrangian one-step

Implement Stooke et al. (2020) PID-Lagrangian on $\lambda$.

> **Hint**: $\lambda \leftarrow \max(0, K_p e + K_i \sum_t e_t + K_d (J_c^{(t)} - J_c^{(t-1)}))$.

In [ ]:
def exercise_8(lam_prev, integral_prev, J_c_hist, d, Kp=0.1, Ki=0.05, Kd=0.05):
    # YOUR CODE HERE
    return ref.exercise_8(lam_prev, integral_prev, J_c_hist, d, Kp, Ki, Kd)

In [ ]:
lam, integ = exercise_8(0.0, 0.0, [2.0, 2.5], 1.0)
assert lam > 0.0 and integ > 0.0
print('lam', lam, 'integ', integ)

<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

Attempt the exercise first. Before opening a hint, write the quantity you are
estimating, the data source, and the expected update direction. The
[teaching guide](teaching_guide.md) supplies worked examples, assumptions,
and transfer questions. The hints below collect the exercise's own inline
hints; they do not reveal or execute reference solutions.

<details>
<summary>Exercise 1 ★ — Lagrangian dual step</summary>

> **Hint** (try yourself first): descent on this Lagrangian gives $\lambda \leftarrow [\lambda + \eta(J_c - d)]_+$. Sign matters.

</details>

<details>
<summary>Exercise 2 ★ — P3O hinge subgradient</summary>

> **Hint**: it is $\kappa$ above $d$, $0$ below $d$, anything in $[0,\kappa]$ at the kink.

</details>

<details>
<summary>Exercise 3 ★★ — GAE on the cost stream</summary>

> **Hint**: iterate from the end; $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ and $A_t = \delta_t + \gamma\lambda A_{t+1}$ when nonterminal.

</details>

<details>
<summary>Exercise 4 ★★ — FOCOPS closed-form non-parametric update</summary>

> **Hint**: log-space + softmax for numerical stability.

</details>

<details>
<summary>Exercise 5 ★★ — Empirical CVaR</summary>

> **Hint**: sort, take the worst $\lceil \alpha N \rceil$, average.

</details>

<details>
<summary>Exercise 6 ★★ — Meaningful objective: train PPO-Lagrangian and audit</summary>

> **Hint**: call `ref.exercise_6(n_updates=20, cost_limit=1.5)`.

</details>

<details>
<summary>Exercise 7 ★★★ — Open-ended: safe vs unsafe</summary>

> **Hint**: `ref.exercise_7` returns the two histories.

</details>

<details>
<summary>Exercise 8 ★★★ — PID-Lagrangian one-step</summary>

> **Hint**: $\lambda \leftarrow \max(0, K_p e + K_i \sum_t e_t + K_d (J_c^{(t)} - J_c^{(t-1)}))$.

</details>

After solving, change one input or assumption and predict the result before
running it. If a cell calls `project.solutions`, passing it verifies the
reference demonstration; test your own implementation separately.
